# Entrenamiento de modelos - Taller Desarrollo en Contenedores

Este notebook entrena un clasificador de **scikit-learn** y guarda cada
modelo entrenado en el volumen **compartido** con el servicio `api`
(carpeta `MODELS_DIR`, montada como `./models` en el host).

Cada vez que ejecutas la celda de entrenamiento se guarda un **nuevo**
archivo de modelo con timestamp (`model_<timestamp>.joblib`) junto con
sus metadatos (`model_<timestamp>.json`). La API detecta automáticamente
el modelo más reciente en su siguiente petición, **sin necesidad de
reiniciar el contenedor de la API**.

In [ ]:
import os
import json
import time
from datetime import datetime

import joblib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

MODELS_DIR = os.environ.get("MODELS_DIR", "/app/models")
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Los modelos se guardarán en: {MODELS_DIR}")

## 1. Cargar y dividir el dataset

Usamos el dataset clásico *iris* (incluido en scikit-learn, sin dependencias externas).

In [ ]:
dataset = load_iris()
X, y = dataset.data, dataset.target
feature_names = list(dataset.feature_names)
target_names = list(dataset.target_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Función de entrenamiento y guardado versionado

Guarda el modelo (`.joblib`) y sus metadatos (`.json`) con un timestamp único.

In [ ]:
def train_and_save(n_estimators=100, max_depth=None, random_state=42):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    timestamp = datetime.utcnow().strftime("%Y%m%d%H%M%S")
    model_filename = f"model_{timestamp}.joblib"
    meta_filename = f"model_{timestamp}.json"

    model_path = os.path.join(MODELS_DIR, model_filename)
    meta_path = os.path.join(MODELS_DIR, meta_filename)

    joblib.dump(model, model_path)

    metadata = {
        "timestamp": timestamp,
        "created_at": datetime.utcnow().isoformat() + "Z",
        "algorithm": "RandomForestClassifier",
        "hyperparameters": {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "random_state": random_state,
        },
        "accuracy": accuracy,
        "feature_names": feature_names,
        "target_names": target_names,
        "model_file": model_filename,
    }
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Modelo guardado: {model_path}")
    print(f"Metadatos guardados: {meta_path}")
    print(f"Accuracy en test: {accuracy:.4f}")
    return model_path, metadata

## 3. Entrenar el primer modelo

In [ ]:
train_and_save(n_estimators=100, max_depth=None)

## 4. Entrenar nuevos modelos (versionado)

Cada vez que corras esta celda (o la anterior) con hiperparámetros
distintos, se generará un **nuevo** archivo de modelo en el volumen
compartido. La API, al recibir la siguiente petición a `/predict`
(o al llamar a `/reload`), detecta el archivo más reciente por fecha
de modificación y lo carga automáticamente.

In [ ]:
# Ejemplo: un segundo modelo con otros hiperparámetros
train_and_save(n_estimators=200, max_depth=5)

## 5. Ver todos los modelos guardados en el volumen compartido

In [ ]:
saved = sorted(
    [f for f in os.listdir(MODELS_DIR) if f.endswith(".joblib")],
    key=lambda f: os.path.getmtime(os.path.join(MODELS_DIR, f)),
)
for f in saved:
    mtime = time.ctime(os.path.getmtime(os.path.join(MODELS_DIR, f)))
    print(f"{f}  (modificado: {mtime})")

if saved:
    print(f"\nModelo más reciente (el que consumirá la API): {saved[-1]}")